# Solution 11 — Support triage on one budget

A router in front of a billing and a tech specialist. Every agent spends from one budget, the billing agent
alone can see the customer's account, and the last cell proves it all offline with `assert`s.

In [ ]:
from learningdantic import setup

MODEL = setup()

## The agents

`lookup_account` is a plain `def`: it reads `ctx.deps` and runs no agent. `to_billing` and `to_tech` await
an agent run, so they are `async def`, and each forwards `usage` and `usage_limits` (plus `deps` for billing).

In [ ]:
from dataclasses import dataclass

from pydantic import BaseModel
from pydantic_ai import Agent, RunContext, UsageLimits

from learningdantic import has_key

LIVE = has_key(MODEL)  # True if MODEL's key (or login) is set up. It sends no request.
# No key: build on the fake 'test' model so Part 2 still runs (it swaps in scripted models anyway).
AGENT_MODEL = MODEL if LIVE else "test"


@dataclass
class Account:
    name: str
    plan: str
    last_charge: str


billing_agent = Agent(
    AGENT_MODEL,
    name="billing",
    deps_type=Account,
    instructions=(
        "You are billing support. Call lookup_account before answering. Answer in at most two sentences."
    ),
)


@billing_agent.tool
def lookup_account(ctx: RunContext[Account]) -> str:
    """Look up the customer's plan and last charge."""
    a = ctx.deps
    return f"{a.name} is on the {a.plan} plan; last charge: {a.last_charge}."


tech_agent = Agent(
    AGENT_MODEL,
    name="tech",
    instructions="You are technical support. Give one concrete next step in at most two sentences.",
)


class Escalate(BaseModel):
    """Use when the message is not about billing or tech, or the customer asks for a human."""

    reason: str


async def to_billing(ctx: RunContext[Account], question: str) -> str:
    """Invoices, charges, refunds, plans and prices."""
    answer = await billing_agent.run(
        question, deps=ctx.deps, usage=ctx.usage, usage_limits=ctx.usage_limits
    )
    return f"[billing] {answer.output}"


async def to_tech(ctx: RunContext[Account], question: str) -> str:
    """Bugs, error messages, logins, crashes and setup problems."""
    answer = await tech_agent.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return f"[tech] {answer.output}"


triage = Agent(
    AGENT_MODEL,
    name="triage",
    deps_type=Account,
    output_type=[to_billing, to_tech, Escalate],
    instructions="Send the customer's message to the right desk. Pass the message on word for word.",
)

ada = Account(name="Ada", plan="Pro", last_charge="$40 on 1 October")
BUDGET = UsageLimits(request_limit=5)

## Part 1 — live: each ticket lands on the right desk

This part needs your key. With no key it prints one line and skips; Part 2 still runs.

In [ ]:
if LIVE:
    billing = await triage.run("Why was I charged $40?", deps=ada, usage_limits=BUDGET)
    tech = await triage.run("My password reset email never arrives.", deps=ada, usage_limits=BUDGET)

    for r in (billing, tech):
        print(r.output, "| requests:", r.usage.requests)

    assert str(billing.output).startswith("[billing]"), billing.output
    assert str(tech.output).startswith("[tech]"), tech.output
    assert billing.usage.requests >= 2, billing.usage  # triage + billing, both on one bill
    print("live routing ok")
else:
    print(f"skipped: no key for {MODEL} (Part 2 below runs keyless)")

## Part 2 — offline: routing, one bill, one budget

Scripted models replace all three agents, so this cell needs no key and gives the same answer every time.
The fake billing model calls `lookup_account`, then repeats what the tool returned.

In [ ]:
from pydantic_ai import RunUsage, UsageLimitExceeded
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel


def fake_triage(messages, info):
    """Scripted router: messages about a charge go to billing, everything else to tech."""
    text = messages[-1].parts[-1].content  # the customer's message
    desk = "final_result_to_billing" if "charged" in text else "final_result_to_tech"
    return ModelResponse(parts=[ToolCallPart(desk, {"question": text})])


def fake_billing(messages, info):
    """Scripted billing agent: look the account up first, then answer with what came back."""
    found = [p.content for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
    if not found:
        return ModelResponse(parts=[ToolCallPart("lookup_account", {})])
    return ModelResponse(parts=[TextPart(f"Found it: {found[0]}")])


def fake_tech(messages, info):
    return ModelResponse(parts=[TextPart("Check your spam folder.")])


with (
    triage.override(model=FunctionModel(fake_triage)),
    billing_agent.override(model=FunctionModel(fake_billing)),
    tech_agent.override(model=FunctionModel(fake_tech)),
):
    r = await triage.run("Why was I charged $40?", deps=ada, usage_limits=BUDGET)
    assert (
        r.output == "[billing] Found it: Ada is on the Pro plan; last charge: $40 on 1 October."
    ), r.output
    assert r.usage.requests == 3, r.usage  # triage 1 + billing 2, all on one bill

    r = await triage.run("My password reset email never arrives.", deps=ada, usage_limits=BUDGET)
    assert r.output == "[tech] Check your spam folder.", r.output
    assert r.usage.requests == 2, r.usage  # triage 1 + tech 1

    bill = RunUsage()
    try:
        await triage.run(
            "Why was I charged $40?",
            deps=ada,
            usage=bill,
            usage_limits=UsageLimits(request_limit=2),
        )
    except UsageLimitExceeded:
        pass
    else:
        raise AssertionError("the shared budget did not stop the billing agent")
    # billing stopped at the cap instead of spending a third request
    assert bill.requests == 2, bill

print("offline checks ok")